In [1]:
from numpy import *
from ngsolve import *
from netgen.occ import *
from netgen.meshing import MeshPoint
from netgen.meshing import Point3d
from ngsolve.webgui import Draw

Coil_1 = Circle(Pnt(-0.2, 0), 0.1).Face()
Coil_2 = Circle(Pnt( 0.2, 0), 0.1).Face()
Circle_1 = Circle(Pnt(0.0, 0), 1.0).Face()
Circle_2 = Circle(Pnt(2.5, 0), 1.0).Face()

cutline = []
#cutline.append(Edge(Vertex(Pnt(2.5, -1.0, 0)), Vertex(Pnt(2.5, 1.0, 0))))
#cutline.append(Edge(Vertex(Pnt(2.0, -1.0, 0)), Vertex(Pnt(2.0, 1.0, 0))))
#cutline.append(Edge(Vertex(Pnt(3.0, -1.0, 0)), Vertex(Pnt(3.0, 1.0, 0))))

vertex = Vertex(Pnt(2.5,0,0))
vertex.name = "GND"
shape = Glue([Coil_1, Coil_2, Circle_1, Circle_2, vertex, *cutline])

shape.faces[0].name = "coil_1"
shape.faces[1].name = "coil_2"
shape.faces[2].name = "air"
shape.faces[3].name = "air"
shape.faces[2].edges[0].Identify(shape.faces[3].edges[0], "periodic",  IdentificationType.PERIODIC)

geo = OCCGeometry(shape, dim=2)
ngmesh = geo.GenerateMesh(maxh=0.3, grading=0.4)
mesh = Mesh(ngmesh).Curve(3)
#Draw(mesh, order=2)

print(f'{len(mesh.GetMaterials())} Materials {mesh.GetMaterials()}')
print(f'{len(mesh.GetBoundaries())} Boundaries {mesh.GetBoundaries()}')
print(f'{len(mesh.GetBBoundaries())} BBoundaries {mesh.GetBBoundaries()}')
print(f'{len(mesh.GetBBBoundaries())} BBBoundaries {mesh.GetBBBoundaries()}')

fes = H1(mesh, order=3, dirichlet_bbnd="GND")
fes = Periodic(fes)
A,N = fes.TnT()

gfA = GridFunction(fes)
mu0 = 1/(4*pi*1e-7)

J = mesh.MaterialCF({"coil_1":1, "coil_2":-1}, default=0)
with TaskManager():
	a = BilinearForm(fes)
	a += 1/mu0*grad(A)*grad(N)*dx
	f = LinearForm(fes)
	f += N*J*dx
	a.Assemble()
	f.Assemble()
with TaskManager():
	gfA.vec.data = a.mat.Inverse(fes.FreeDofs()) * f.vec
Draw(gfA, mesh, "B-field", vectors={"grid_size":50})

B = CoefficientFunction((grad(gfA)[1], -grad(gfA)[0]))
Draw(B, mesh, "B", vectors={"grid_size":50})



4 Materials ('coil_1', 'coil_2', 'air', 'air')
4 Boundaries ('default', 'default', 'default', 'default')
5 BBoundaries ('default', 'default', 'default', 'default', 'GND')
0 BBBoundaries ()


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [2]:
#help(Pnt(3,0))
help(Vertex)

Help on class Vertex in module netgen.libngpy._NgOCC:

class Vertex(TopoDS_Shape)
 |  Method resolution order:
 |      Vertex
 |      TopoDS_Shape
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(...)
 |      __init__(*args, **kwargs)
 |      Overloaded function.
 |      
 |      1. __init__(self: netgen.libngpy._NgOCC.Vertex, arg0: netgen.libngpy._NgOCC.TopoDS_Shape) -> None
 |      
 |      2. __init__(self: netgen.libngpy._NgOCC.Vertex, arg0: netgen.libngpy._NgOCC.gp_Pnt) -> None
 |  
 |  ----------------------------------------------------------------------
 |  Readonly properties defined here:
 |  
 |  p
 |      coordinates of vertex
 |  
 |  ----------------------------------------------------------------------
 |  Methods inherited from TopoDS_Shape:
 |  
 |  Distance(...)
 |      Distance(self: netgen.libngpy._NgOCC.TopoDS_Shape, arg0: netgen.libngpy._NgOCC.TopoDS_Shape) -> float
 |  
 |  Extrude(...)
 |      Ext